# AAAI 2026 Nesy-Gen Revision Experiments

This Colab notebook runs the switchable paper-grade pipeline for IU-Xray/R2Gen-IU and MIMIC-CXR:

- R2Gen-style image-to-report generator
- RAG/retrieval report candidates
- PrimeKG entity grounding
- optional soft graph-constrained decoding
- LTN-style ante-hoc graph reasoning
- Consistency Gate candidate selection
- entity-linking validation
- official metric input preparation

Set `RUN_DATASET` and `DECODING_MODE` in Cell 2.

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Cell 2: Main Configuration

In [ ]:
from pathlib import Path

RUN_DATASET = "r2gen_iuxray"  # options: "r2gen_iuxray", "mimic_aug"
DECODING_MODE = "graph_constrained"  # options: "standard", "graph_constrained"

RUN_NAME = "aaai_rag_primekg_ltn_gate"
SEED = 13
SMOKE_RUN = True

REPO_DIR = Path("/content/Nesy-GenRevision")
PRIMEKG_SOURCE_DIR = Path("/content/drive/MyDrive/dataverse_files")

AAAI_ROOT = Path("/content/drive/MyDrive/aaai_2026_experiments")
OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAD_PRIMEKG_DIR = Path(f"/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}")

# Smoke training, not final paper training.
# If this works, increase MAX_TRAIN_EXAMPLES to 1000, then set SMOKE_RUN=False/full split.
MAX_TRAIN_EXAMPLES = 256 if SMOKE_RUN else 1000
MAX_VAL_EXAMPLES = 64 if SMOKE_RUN else 256
MAX_TEST_EXAMPLES = 100 if SMOKE_RUN else None

RETRIEVAL_TOP_K = 5
R2GEN_NUM_CANDIDATES = 4
VISUAL_SEQ_LEN = 128

# Soft graph-constrained decoding settings.
GRAPH_TOKEN_BOOST = 2.0
UNSUPPORTED_TOKEN_PENALTY = 0.0
CONSTRAINT_MAX_TERMS = 2500

# BLEU-oriented graph-constrained setting.
# Use "graph" for strict factuality, "hybrid" to balance BLEU/retrieval evidence and graph consistency.
SELECTION_OBJECTIVE = "hybrid" if DECODING_MODE == "graph_constrained" else "graph"
GRAPH_SCORE_WEIGHT = 0.45
EVIDENCE_WEIGHT = 0.45
GATE_WEIGHT = 0.10
GENERATED_EVIDENCE_SCORE = 0.50

GENERATOR_NAME = "rag_primekg_constrained" if DECODING_MODE == "graph_constrained" else "rag_primekg"

print("Dataset:", RUN_DATASET)
print("Decoding:", DECODING_MODE)
print("Generator:", GENERATOR_NAME)
print("Output:", OUTPUT_DIR)

## Cell 3: Clone/Pull Repo And Install Dependencies

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/FaezehMillerAI/Nesy-GenRevision.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[torch,eval,colab]"],
    check=True,
)

print("Repo ready:", REPO_DIR)

## Cell 4: GPU Check

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 5: Resolve Dataset Root

In [ ]:
from pathlib import Path

if RUN_DATASET == "r2gen_iuxray":
    DATA_ROOT = Path("/content/drive/MyDrive/iuxray")
    MANIFEST = OUTPUT_DIR / "r2gen_iuxray_manifest.jsonl"

elif RUN_DATASET == "mimic_aug":
    MIMIC_DRIVE_ROOT = Path("/content/drive/MyDrive/mimic_cxr_dataset/versions/2")
    if MIMIC_DRIVE_ROOT.exists():
        DATA_ROOT = MIMIC_DRIVE_ROOT
    else:
        import kagglehub
        DATA_ROOT = Path(kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset"))
        while not (DATA_ROOT / "mimic_cxr_aug_train.csv").exists() and DATA_ROOT.parent != DATA_ROOT:
            DATA_ROOT = DATA_ROOT.parent
    MANIFEST = OUTPUT_DIR / "mimic_aug_manifest.jsonl"

else:
    raise ValueError(RUN_DATASET)

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())
print("MANIFEST:", MANIFEST)

## Cell 6: Build Dataset Manifest

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "scripts/build_manifest.py",
        "--dataset", RUN_DATASET,
        "--data-root", str(DATA_ROOT),
        "--output", str(MANIFEST),
        "--seed", str(SEED),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Manifest exists:", MANIFEST.exists())

## Cell 7: Inspect Manifest

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in MANIFEST.read_text().splitlines()]
print("Examples:", len(rows))
print("Splits:", Counter(row["split"] for row in rows))
rows[0]

## Cell 8: Build Or Reuse Radiology PrimeKG Cache

In [ ]:
import subprocess
import sys

if not (RAD_PRIMEKG_DIR / "kg.csv").exists() or not (RAD_PRIMEKG_DIR / "nodes.csv").exists():
    subprocess.run(
        [
            sys.executable, "scripts/build_radiology_primekg.py",
            "--primekg-dir", str(PRIMEKG_SOURCE_DIR),
            "--manifest", str(MANIFEST),
            "--output-dir", str(RAD_PRIMEKG_DIR),
            "--hops", "1",
        ],
        cwd=REPO_DIR,
        check=True,
    )

print("PrimeKG cache:", RAD_PRIMEKG_DIR)
print((RAD_PRIMEKG_DIR / "radiology_primekg_summary.json").read_text())

## Cell 9: Entity Extraction/Linking Validation

In [ ]:
import subprocess
import sys

ENTITY_VALIDATION_DIR = OUTPUT_DIR / "entity_linking_validation"

subprocess.run(
    [
        sys.executable, "scripts/validate_entity_linking.py",
        "--manifest", str(MANIFEST),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
        "--output-dir", str(ENTITY_VALIDATION_DIR),
        "--prefix", RUN_DATASET,
        "--split", "test",
        "--limit", str(MAX_TEST_EXAMPLES or 1000),
        "--audit-sample-size", "100",
        "--seed", str(SEED),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Entity validation:", ENTITY_VALIDATION_DIR)

## Cell 10: Train R2Gen-T5 Generator

In [ ]:
import subprocess
import sys

R2GEN_CKPT = OUTPUT_DIR / "checkpoints" / "r2gen_t5_resnet101_t5small"

cmd = [
    sys.executable, "-u", "scripts/train_r2gen_t5_generator.py",
    "--manifest", str(MANIFEST),
    "--output-dir", str(R2GEN_CKPT),
    "--text-model-name", "t5-small",
    "--tokenizer-name", "t5-small",
    "--epochs", "1" if SMOKE_RUN else "3",
    "--batch-size", "2",
    "--gradient-accumulation-steps", "2",
    "--learning-rate", "1e-4",
    "--max-target-length", "160",
    "--visual-seq-len", str(VISUAL_SEQ_LEN),
    "--seed", str(SEED),
    "--fp16",
]

if MAX_TRAIN_EXAMPLES:
    cmd += ["--max-train-examples", str(MAX_TRAIN_EXAMPLES)]
if MAX_VAL_EXAMPLES:
    cmd += ["--max-val-examples", str(MAX_VAL_EXAMPLES)]

subprocess.run(cmd, cwd=REPO_DIR, check=True)
print("Checkpoint:", R2GEN_CKPT)

## Cell 11: Run RAG + PrimeKG + LTN + Consistency Gate

In [ ]:
import subprocess
import sys

RAG_OUTPUT_CSV = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_predictions.csv"
RAG_CANDIDATES_CSV = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_candidate_audit.csv"

cmd = [
    sys.executable, "-u", "scripts/generate_rag_primekg_reports.py",
    "--manifest", str(MANIFEST),
    "--primekg-dir", str(RAD_PRIMEKG_DIR),
    "--output-csv", str(RAG_OUTPUT_CSV),
    "--candidates-csv", str(RAG_CANDIDATES_CSV),
    "--split", "test",
    "--retrieval-top-k", str(RETRIEVAL_TOP_K),
    "--r2gen-checkpoint-dir", str(R2GEN_CKPT),
    "--r2gen-num-candidates", str(R2GEN_NUM_CANDIDATES),
    "--generated-evidence-score", str(GENERATED_EVIDENCE_SCORE),
    "--r2gen-num-beams", "6",
    "--r2gen-batch-size", "2",
    "--max-new-tokens", "120",
    "--decoding-mode", DECODING_MODE,
    "--graph-token-boost", str(GRAPH_TOKEN_BOOST),
    "--unsupported-token-penalty", str(UNSUPPORTED_TOKEN_PENALTY),
    "--constraint-max-terms", str(CONSTRAINT_MAX_TERMS),
    "--subgraph-strategy", "ego",
    "--min-graph-score", "0.50",
    "--selection-objective", SELECTION_OBJECTIVE,
    "--graph-score-weight", str(GRAPH_SCORE_WEIGHT),
    "--evidence-weight", str(EVIDENCE_WEIGHT),
    "--gate-weight", str(GATE_WEIGHT),
]

if MAX_TEST_EXAMPLES:
    cmd += ["--limit", str(MAX_TEST_EXAMPLES)]

subprocess.run(cmd, cwd=REPO_DIR, check=True)

print("Predictions:", RAG_OUTPUT_CSV)
print("Candidate audit:", RAG_CANDIDATES_CSV)

## Cell 12: Quick Internal Evaluation

This gives fast development feedback. For final paper numbers, use the official outputs prepared in Cell 13 and evaluated in Cell 14.

In [ ]:
import subprocess
import sys

EVAL_JSON = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_quick_eval.json"

subprocess.run(
    [
        sys.executable, "scripts/evaluate_generation.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(EVAL_JSON),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
        "--output-factuality-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_entity_factuality.csv"),
        "--output-chexpert-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_chexpert_lite.csv"),
        "--output-radgraph-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_radgraph_lite.csv"),
    ],
    cwd=REPO_DIR,
    check=True,
)

print(EVAL_JSON.read_text())

## Cell 13: Leakage / Reference-Overlap Audit

Run this before trusting unusually high BLEU/ROUGE scores.


In [ ]:
import subprocess
import sys

LEAKAGE_AUDIT_JSON = OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_leakage_audit.json"

subprocess.run(
    [
        sys.executable, "scripts/audit_prediction_leakage.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(LEAKAGE_AUDIT_JSON),
        "--high-overlap-threshold", "0.95",
    ],
    cwd=REPO_DIR,
    check=True,
)

print(LEAKAGE_AUDIT_JSON.read_text())

## Cell 14: Prepare Official Metric Inputs

In [ ]:
import subprocess
import sys

OFFICIAL_INPUT_DIR = OUTPUT_DIR / f"official_metric_inputs_{GENERATOR_NAME}"

subprocess.run(
    [
        sys.executable, "scripts/evaluate_generation.py",
        "--manifest", str(MANIFEST),
        "--predictions-csv", str(RAG_OUTPUT_CSV),
        "--output-json", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_input_prep.json"),
        "--prepare-official-inputs-dir", str(OFFICIAL_INPUT_DIR),
        "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
    ],
    cwd=REPO_DIR,
    check=True,
)

print("Official inputs:", OFFICIAL_INPUT_DIR)
print("CheXbert pred:", OFFICIAL_INPUT_DIR / "pred_reports_for_chexbert.csv")
print("CheXbert ref:", OFFICIAL_INPUT_DIR / "ref_reports_for_chexbert.csv")
print("RadGraph input:", OFFICIAL_INPUT_DIR / "reports_for_radgraph.json")

## Cell 15: Official Evaluation After External CheXbert/RadGraph

Run the official CheXbert/CheXpert and RadGraph tools on the files from Cell 13, save their outputs into `OFFICIAL_OUTPUT_DIR`, then rerun this cell.

In [ ]:
import subprocess
import sys
from pathlib import Path

OFFICIAL_OUTPUT_DIR = OUTPUT_DIR / f"official_metric_outputs_{GENERATOR_NAME}"

CHEXBERT_PRED_LABELS_CSV = OFFICIAL_OUTPUT_DIR / "pred_chexbert_labels.csv"
CHEXBERT_REF_LABELS_CSV = OFFICIAL_OUTPUT_DIR / "ref_chexbert_labels.csv"
RADGRAPH_PRED_JSON = OFFICIAL_OUTPUT_DIR / "pred_radgraph.json"
RADGRAPH_REF_JSON = OFFICIAL_OUTPUT_DIR / "ref_radgraph.json"

required = [
    CHEXBERT_PRED_LABELS_CSV,
    CHEXBERT_REF_LABELS_CSV,
    RADGRAPH_PRED_JSON,
    RADGRAPH_REF_JSON,
]

if not all(path.exists() for path in required):
    print("Official external outputs are not ready yet.")
    for path in required:
        print(path, path.exists())
else:
    subprocess.run(
        [
            sys.executable, "scripts/evaluate_generation.py",
            "--manifest", str(MANIFEST),
            "--predictions-csv", str(RAG_OUTPUT_CSV),
            "--output-json", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_metrics.json"),
            "--official-coco",
            "--official-only",
            "--chexbert-pred-labels-csv", str(CHEXBERT_PRED_LABELS_CSV),
            "--chexbert-ref-labels-csv", str(CHEXBERT_REF_LABELS_CSV),
            "--radgraph-pred-json", str(RADGRAPH_PRED_JSON),
            "--radgraph-ref-json", str(RADGRAPH_REF_JSON),
            "--output-official-chexbert-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_chexbert.csv"),
            "--output-official-radgraph-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_radgraph.csv"),
            "--nodes-csv", str(RAD_PRIMEKG_DIR / "nodes.csv"),
            "--output-factuality-csv", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_official_entity_factuality.csv"),
        ],
        cwd=REPO_DIR,
        check=True,
    )

## Cell 16: Save Reproducible Experiment Config

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "scripts/write_experiment_config.py",
        "--output", str(OUTPUT_DIR / f"{RUN_DATASET}_{GENERATOR_NAME}_experiment_config.json"),
        "--dataset", RUN_DATASET,
        "--manifest", str(MANIFEST),
        "--primekg-dir", str(RAD_PRIMEKG_DIR),
        "--output-dir", str(OUTPUT_DIR),
        "--run-name", RUN_NAME,
        "--generator", GENERATOR_NAME,
        "--r2gen-checkpoint-dir", str(R2GEN_CKPT),
        "--retrieval-top-k", str(RETRIEVAL_TOP_K),
        "--r2gen-num-candidates", str(R2GEN_NUM_CANDIDATES),
        "--decoding-mode", DECODING_MODE,
        "--graph-token-boost", str(GRAPH_TOKEN_BOOST),
        "--unsupported-token-penalty", str(UNSUPPORTED_TOKEN_PENALTY),
        "--selection-objective", SELECTION_OBJECTIVE,
        "--graph-score-weight", str(GRAPH_SCORE_WEIGHT),
        "--evidence-weight", str(EVIDENCE_WEIGHT),
        "--gate-weight", str(GATE_WEIGHT),
        "--subgraph-strategy", "ego",
        "--official-metrics",
    ],
    cwd=REPO_DIR,
    check=True,
)

## Cell 17: Inspect Outputs

In [ ]:
import pandas as pd

print("Predictions:", RAG_OUTPUT_CSV)
print("Candidates:", RAG_CANDIDATES_CSV)

pred_df = pd.read_csv(RAG_OUTPUT_CSV)
cand_df = pd.read_csv(RAG_CANDIDATES_CSV)

print("Prediction rows:", len(pred_df))
print("Candidate rows:", len(cand_df))
display(pred_df.head())
display(cand_df.head())

## Recommended AAAI Ablations

Run the notebook with these settings:

1. `RUN_DATASET = "r2gen_iuxray"`, `DECODING_MODE = "standard"`
2. `RUN_DATASET = "r2gen_iuxray"`, `DECODING_MODE = "graph_constrained"`
3. `RUN_DATASET = "mimic_aug"`, `DECODING_MODE = "standard"`
4. `RUN_DATASET = "mimic_aug"`, `DECODING_MODE = "graph_constrained"`

For final paper runs, set `SMOKE_RUN = False` and run the full train/test splits.